# Overfitting, Regularization, and Structural Risk

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 13.01, 13.02, 01.04 |
| Optional | 03.05 (Bayesian inference — for the MAP connection) |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/13_Statistical_Learning_Theory/03_overfitting_regularization_structural_risk.ipynb)

---

## 🎯 Learning Objective

Formalize overfitting as a gap between empirical and true risk, derive the two principal
regularization penalties (L2/Ridge and L1/Lasso) and their Bayesian interpretations, and
understand Vapnik’s Structural Risk Minimization principle that balances approximation error
against estimation error by searching over a nested hierarchy of hypothesis classes.

---

## 📐 Theory

This notebook takes the empirical risk minimization (ERM) framework from `13.01` and the
generalization gap from `13.02` and asks the central question of statistical learning:
how do we choose a hypothesis class that is rich enough to approximate the true target
function, yet constrained enough that the finite training sample does not mislead us?

### 1 — Overfitting, formalized

Recall from `13.01` the true risk $R(h) = \mathbb{E}_{(x,y)\sim\mathcal{D}}[\ell(h(x), y)]$
and the empirical risk $\hat{R}_n(h) = \frac{1}{n}\sum_{i=1}^{n}\ell(h(x_i), y_i)$. The
ERM solution over a hypothesis class $\mathcal{H}$ is
$\hat{h} = \arg\min_{h\in\mathcal{H}} \hat{R}_n(h)$.

**Definition (Overfitting).** We say $\hat{h}$ *overfits* if the **estimation error** is
large relative to the **approximation error**:

$$R(\hat{h}) - \inf_{h\in\mathcal{H}} R(h) \;\gg\; \inf_{h\in\mathcal{H}} R(h) - R^*.$$

Here $R^* = \inf_{h:\mathcal{X}\to\mathcal{Y}} R(h)$ is the Bayes risk — the irreducible
error. The first term measures how much worse $\hat{h}$ is than the *best* function
in $\mathcal{H}$ (this is the estimation error, which depends on sample size $n$ and
the complexity of $\mathcal{H}$). The second term measures how far the best function
in $\mathcal{H}$ is from the globally optimal predictor (the approximation error, which
depends only on $\mathcal{H}$, not on the data).

The **bias–complexity trade-off** is the tension between these two terms. Making
$\mathcal{H}$ larger reduces the approximation error (lower bias) but inflates the
estimation error (higher variance / overfitting). The uniform convergence bound from
`13.02` makes this precise: with probability at least $1-\delta$,

$$R(\hat{h}) \le \hat{R}_n(\hat{h}) + \epsilon_n(\mathcal{H}, \delta),$$

where $\epsilon_n$ grows with the complexity of $\mathcal{H}$ (measured by VC dimension
in `13.05`, or Rademacher complexity in `13.06`).

### 2 — Regularized Empirical Risk Minimization

Rather than restricting $\mathcal{H}$ outright, regularized ERM adds a penalty
$\Omega(h)$ that discourages complex hypotheses:

$$\hat{h}_\lambda = \arg\min_{h\in\mathcal{H}} \Big[\hat{R}_n(h) + \lambda\,\Omega(h)\Big],$$

where $\lambda \ge 0$ is the **regularization strength**. When $\lambda=0$ we recover
unregularized ERM; as $\lambda \to \infty$, the penalty dominates and the solution is
driven toward the simplest hypothesis in $\mathcal{H}$ (e.g., $w=0$ for norm-based
penalties).

#### 2a — L2 regularization (Ridge / Tikhonov)

Set $\Omega(w) = \|w\|_2^2 = \sum_j w_j^2$. For the squared-error loss
$\ell(\hat{y},y)=(\hat{y}-y)^2$ and a linear model $h_w(x)=w^\top x$,
the objective becomes:

$$\hat{w}_\lambda^{\text{Ridge}} = \arg\min_w \frac{1}{n}\|Xw - y\|_2^2 + \lambda\|w\|_2^2.$$

Taking the gradient and setting it to zero (standard matrix calculus from `01.04`):

$$\nabla_w = \frac{2}{n}X^\top(Xw - y) + 2\lambda w = 0$$

$$\implies \hat{w}_\lambda^{\text{Ridge}} = (X^\top X + n\lambda I)^{-1} X^\top y.$$

**Geometric interpretation via SVD.** Let $X = U\Sigma V^\top$ be the SVD (`01.04`).
Then:

$$\hat{w}_\lambda^{\text{Ridge}} = V \,\text{diag}\!\left(\frac{\sigma_j}{\sigma_j^2 + n\lambda}\right) U^\top y,$$

where $\sigma_j$ are the singular values. Each singular component is *shrunk* by the
factor $\sigma_j/(\sigma_j^2 + n\lambda)$ compared to the OLS solution's
$1/\sigma_j$. Components with small singular values (noisy directions) are shrunk
most aggressively — this is how Ridge *stabilizes* an ill-conditioned design matrix.

#### 2b — L1 regularization (Lasso)

Set $\Omega(w) = \|w\|_1 = \sum_j |w_j|$:

$$\hat{w}_\lambda^{\text{Lasso}} = \arg\min_w \frac{1}{n}\|Xw - y\|_2^2 + \lambda\|w\|_1.$$

The L1 norm is **not differentiable** at $w_j=0$, so there is no closed-form solution.
However, the subdifferential condition is:

$$\frac{2}{n}X_j^\top(Xw - y) + \lambda\,\partial|w_j| \ni 0,$$

where $\partial|w_j| = \{\text{sign}(w_j)\}$ if $w_j\ne 0$ and $\partial|w_j| = [-1,1]$
if $w_j=0$. The key consequence: for any coordinate $j$ where the gradient of the
data-fit term has magnitude less than $\lambda$, the optimality condition is satisfied
at $w_j=0$ exactly. This is why L1 **promotes sparsity** — it drives small coefficients
to exact zeros, performing automatic feature selection.

The standard algorithm is **ISTA** (Iterative Shrinkage-Thresholding Algorithm), a
proximal gradient method:

$$w^{(t+1)} = \text{prox}_{\eta\lambda\|\cdot\|_1}\!\Big(w^{(t)} - \eta\,\nabla_w \hat{R}_n(w^{(t)})\Big),$$

where $\eta$ is the step size and the **proximal operator** (soft-thresholding) acts
coordinate-wise:

$$\text{prox}_{\eta\lambda\|\cdot\|_1}(z)_j = \text{sign}(z_j)\max(|z_j|-\eta\lambda,\, 0).$$

#### 2c — Elastic Net

Combines both penalties:

$$\Omega(w) = \alpha\|w\|_1 + (1-\alpha)\|w\|_2^2, \quad \alpha\in[0,1].$$

This inherits L1's sparsity while avoiding L1's instability when features are
highly correlated (the L2 term encourages grouped selection).

### 3 — Bayesian interpretation: regularization as a prior

From `03.05` (Bayesian inference), the MAP estimate is:

$$\hat{w}_{\text{MAP}} = \arg\max_w \log p(\mathcal{D}\mid w) + \log p(w).$$

**L2 ↔ Gaussian prior.** If $p(w) = \mathcal{N}(0, \tau^2 I)$, then
$\log p(w) = -\frac{1}{2\tau^2}\|w\|_2^2 + \text{const}$. With a Gaussian likelihood
(squared loss), the negative log-posterior becomes:

$$-\log p(w\mid\mathcal{D}) \propto \frac{1}{2\sigma^2}\|Xw-y\|_2^2 + \frac{1}{2\tau^2}\|w\|_2^2.$$

Minimizing this is *identical* to Ridge regression with
$\lambda = \sigma^2/(n\tau^2)$. Large prior variance $\tau^2$ (weak prior)
corresponds to small $\lambda$ (less regularization), and vice versa.

**L1 ↔ Laplace prior.** If $p(w_j) = \frac{1}{2b}\exp(-|w_j|/b)$ (Laplace
distribution), then $\log p(w) = -\frac{1}{b}\|w\|_1 + \text{const}$. Minimizing
the negative log-posterior gives the Lasso with $\lambda = \sigma^2/(nb)$.

The Laplace prior has a sharp peak at zero and heavy tails — it encodes the belief
that most weights should be zero (sparse) but a few can be large, exactly the
inductive bias Lasso imposes.

### 4 — Structural Risk Minimization (SRM)

Vapnik's **Structural Risk Minimization** principle provides a principled alternative
to choosing $\lambda$ by cross-validation. The idea:

1. Construct a **nested hierarchy** of hypothesis classes of increasing complexity:
   $\mathcal{H}_1 \subset \mathcal{H}_2 \subset \mathcal{H}_3 \subset \cdots$

   For example, polynomials of degree $\le 1 \subset$ degree $\le 2 \subset \cdots$,
   or neural networks with $\le k$ parameters.

2. For each $\mathcal{H}_k$, compute the ERM solution $\hat{h}_k$ and its
   generalization bound:

   $$R(\hat{h}_k) \le \hat{R}_n(\hat{h}_k) + \text{pen}(\mathcal{H}_k, n, \delta),$$

   where $\text{pen}$ is a complexity penalty — e.g. the VC-dimension-based bound
   (→ `13.05`):

   $$\text{pen}(\mathcal{H}_k, n, \delta) = \sqrt{\frac{d_{\text{VC}}(\mathcal{H}_k)(\ln(2n/d_{\text{VC}}(\mathcal{H}_k)) + 1) + \ln(4/\delta)}{n}},$$

   or data-dependent Rademacher bounds (→ `13.06`).

3. **Select** $k^* = \arg\min_k \big[\hat{R}_n(\hat{h}_k) + \text{pen}(\mathcal{H}_k, n, \delta)\big]$.

SRM automates the bias–complexity trade-off: simple classes have low penalty but
high empirical risk (underfitting), complex classes have low empirical risk but high
penalty (overfitting risk), and the optimum balances the two.

**Connection to regularization.** Regularized ERM can be viewed as a *continuous*
version of SRM: instead of a discrete hierarchy indexed by $k$, the regularization
parameter $\lambda$ continuously interpolates between simple (large $\lambda$) and
complex (small $\lambda$) effective hypothesis classes.

### 5 — Implicit regularization

Not all regularization takes the form of an explicit penalty $\lambda\Omega(h)$.
Several standard training techniques act as *implicit* regularizers:

- **Early stopping.** Halting gradient descent before convergence constrains the
  effective complexity of the learned model. For linear regression with gradient
  descent, stopping after $T$ iterations is equivalent to L2 regularization with
  $\lambda \approx 1/(\eta T)$, where $\eta$ is the learning rate.

- **Dropout.** Randomly zeroing activations during training. For a single linear
  layer, dropout with rate $p$ is equivalent to L2 regularization with
  $\lambda \propto p/(1-p)$ (the exact equivalence breaks down for nonlinear networks,
  but the regularizing effect persists).

- **Batch normalization.** Normalizing activations adds noise (through mini-batch
  statistics), which has a regularizing effect similar to adding noise to the
  gradients.

- **Data augmentation.** Artificially expanding the training set with transformed
  copies encodes invariances (e.g., rotation invariance) that constrain the
  effective hypothesis class.

The distinction matters: **explicit** regularization appears in the objective function
and can be analyzed mathematically; **implicit** regularization arises from the
optimization procedure or data pipeline and is often harder to characterize precisely.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Two key visualizations:

**Left & Center — Regularization paths.** We plot Ridge and Lasso coefficient values as a function
of $\lambda$. Ridge shrinks all coefficients smoothly toward zero but never reaches it;
Lasso drives coefficients to exact zero at different $\lambda$ thresholds, performing
automatic feature selection. This is the geometric consequence of the L1 norm's corners
versus the L2 norm's smooth sphere.

**Right — SRM hierarchy diagram.** We show how empirical risk decreases and the complexity
penalty increases as we move along the nested hierarchy $\mathcal{H}_1 \subset \mathcal{H}_2
\subset \cdots$, with the SRM-selected class minimizing their sum (the structural risk bound).

In [ ]:
# --- Visualization: Regularization paths (Ridge vs Lasso) + SRM diagram ---

rng = np.random.default_rng(42)
n, p = 100, 8
X = rng.normal(size=(n, p))
true_w = np.array([3.0, -2.0, 0.0, 0.0, 1.5, 0.0, -0.5, 0.0])  # sparse ground truth
y = X @ true_w + rng.normal(scale=0.5, size=n)

# Ridge path (closed-form)
lambdas = np.logspace(-3, 3, 200)
XtX = X.T @ X
Xty = X.T @ y
ridge_coefs = np.array([np.linalg.solve(XtX + n * lam * np.eye(p), Xty) for lam in lambdas])

# Lasso path via coordinate descent
def lasso_cd(X, y, lam, max_iter=1000, tol=1e-6):
    n, p = X.shape
    w = np.zeros(p)
    for _ in range(max_iter):
        w_old = w.copy()
        for j in range(p):
            r_j = y - X @ w + X[:, j] * w[j]
            rho_j = X[:, j] @ r_j / n
            norm_j = np.sum(X[:, j] ** 2) / n
            w[j] = np.sign(rho_j) * max(abs(rho_j) - lam, 0) / norm_j
        if np.max(np.abs(w - w_old)) < tol:
            break
    return w

lasso_coefs = np.array([lasso_cd(X, y, lam) for lam in lambdas])

colors = ['#4C72B0', '#C44E52', '#55A868', '#8172B2',
          '#CCB974', '#64B5CD', '#E58606', '#8C564B']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ridge path
for j in range(p):
    axes[0].plot(np.log10(lambdas), ridge_coefs[:, j], color=colors[j], lw=1.8,
                label=f'$w_{{{j+1}}}$' if j < 4 else None)
axes[0].axhline(0, color='gray', ls='--', lw=0.8)
axes[0].set_xlabel(r'$\log_{10}(\lambda)$', fontsize=12)
axes[0].set_ylabel('Coefficient value', fontsize=12)
axes[0].set_title('Ridge (L2) Regularization Path', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9, loc='upper right')

# Lasso path
for j in range(p):
    axes[1].plot(np.log10(lambdas), lasso_coefs[:, j], color=colors[j], lw=1.8,
                label=f'$w_{{{j+1}}}$' if j < 4 else None)
axes[1].axhline(0, color='gray', ls='--', lw=0.8)
axes[1].set_xlabel(r'$\log_{10}(\lambda)$', fontsize=12)
axes[1].set_ylabel('Coefficient value', fontsize=12)
axes[1].set_title('Lasso (L1) Regularization Path', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9, loc='upper right')

# SRM diagram
ks = np.arange(1, 11)
emp_risk = 2.5 * np.exp(-0.5 * ks) + 0.05
penalty = 0.15 * np.sqrt(ks)
total = emp_risk + penalty
k_star = np.argmin(total)

axes[2].plot(ks, emp_risk, 'o-', color='#4C72B0', lw=2, ms=6,
            label=r'$\hat{R}_n(\hat{h}_k)$ (empirical risk)')
axes[2].plot(ks, penalty, 's-', color='#C44E52', lw=2, ms=6,
            label=r'$\mathrm{pen}(\mathcal{H}_k, n, \delta)$')
axes[2].plot(ks, total, 'D-', color='#55A868', lw=2.5, ms=7,
            label=r'Structural risk bound (sum)')
axes[2].axvline(ks[k_star], color='#8172B2', ls='--', lw=2,
               label=f'$k^*={ks[k_star]}$ (SRM choice)')
axes[2].set_xlabel(r'Hypothesis class index $k$', fontsize=12)
axes[2].set_ylabel('Risk / Penalty', fontsize=12)
axes[2].set_title('Structural Risk Minimization', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].set_xticks(ks)

plt.tight_layout()
plt.show()

# Report
n_zero_lasso = np.sum(np.abs(lasso_coefs[-1]) < 1e-8)
print(f'At largest lambda, Lasso zeros out {n_zero_lasso}/{p} coefficients (sparsity).')
print(f'Ridge coefficients never reach exactly zero -- they shrink smoothly.')
print(f'SRM selects k* = {ks[k_star]}: empirical risk = {emp_risk[k_star]:.3f}, '
      f'penalty = {penalty[k_star]:.3f}, bound = {total[k_star]:.3f}')

## 🐍 Implementation from Scratch

We implement three core algorithms from scratch using only NumPy:

1. **Ridge regression** via the closed-form solution $(X^\top X + n\lambda I)^{-1}X^\top y$.
2. **Lasso regression** via ISTA (Iterative Shrinkage-Thresholding Algorithm) — the proximal
   gradient method with soft-thresholding as the proximal operator.
3. **Early stopping** demonstration: training a polynomial regression model with gradient
   descent and showing how stopping early is equivalent to implicit L2 regularization.

In [ ]:
# =====================================================================
# 1. Ridge regression -- closed-form solution
# =====================================================================
def ridge_fit(X, y, lam):
    """Closed-form Ridge: w = (X'X + n*lam*I)^{-1} X'y."""
    n, p = X.shape
    return np.linalg.solve(X.T @ X + n * lam * np.eye(p), X.T @ y)

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

rng = np.random.default_rng(7)
n_train, n_test, p = 50, 200, 20
X_train = rng.normal(size=(n_train, p))
X_test = rng.normal(size=(n_test, p))
true_w = np.zeros(p); true_w[:5] = [3, -2, 1.5, -1, 0.5]
y_train = X_train @ true_w + rng.normal(scale=1.0, size=n_train)
y_test = X_test @ true_w + rng.normal(scale=1.0, size=n_test)

# OLS vs Ridge at several lambda values
w_ols = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
print('=' * 65)
print('RIDGE REGRESSION (closed-form)')
print('=' * 65)
print(f'{"lambda":>10} | {"Train MSE":>10} | {"Test MSE":>10} | {"||w||_2":>10}')
print('-' * 50)
print(f'{"OLS":>10} | {mse(y_train, X_train @ w_ols):10.4f} | '
      f'{mse(y_test, X_test @ w_ols):10.4f} | {np.linalg.norm(w_ols):10.4f}')
for lam in [0.001, 0.01, 0.1, 1.0, 10.0]:
    w_r = ridge_fit(X_train, y_train, lam)
    print(f'{lam:10.3f} | {mse(y_train, X_train @ w_r):10.4f} | '
          f'{mse(y_test, X_test @ w_r):10.4f} | {np.linalg.norm(w_r):10.4f}')

# =====================================================================
# 2. Lasso regression -- ISTA (proximal gradient) from scratch
# =====================================================================
def soft_threshold(z, threshold):
    """Proximal operator for L1: sign(z) * max(|z| - threshold, 0)."""
    return np.sign(z) * np.maximum(np.abs(z) - threshold, 0.0)

def lasso_ista(X, y, lam, lr=None, max_iter=2000, tol=1e-6):
    """ISTA for Lasso: gradient step on squared loss, then soft-threshold."""
    n, p = X.shape
    if lr is None:
        L = 2 * np.linalg.norm(X.T @ X / n, ord=2)
        lr = 1.0 / L
    w = np.zeros(p)
    losses = []
    for t in range(max_iter):
        grad = (2.0 / n) * X.T @ (X @ w - y)
        z = w - lr * grad
        w_new = soft_threshold(z, lr * lam)
        obj = (1.0 / n) * np.sum((X @ w_new - y) ** 2) + lam * np.sum(np.abs(w_new))
        losses.append(obj)
        if np.max(np.abs(w_new - w)) < tol:
            break
        w = w_new
    return w, losses

print('\n' + '=' * 65)
print('LASSO REGRESSION (ISTA from scratch)')
print('=' * 65)
print(f'{"lambda":>10} | {"Train MSE":>10} | {"Test MSE":>10} | {"Nonzero":>8} | {"Iters":>6}')
print('-' * 55)
for lam in [0.001, 0.01, 0.1, 0.5, 1.0]:
    w_l, losses_l = lasso_ista(X_train, y_train, lam)
    nnz = np.sum(np.abs(w_l) > 1e-6)
    print(f'{lam:10.3f} | {mse(y_train, X_train @ w_l):10.4f} | '
          f'{mse(y_test, X_test @ w_l):10.4f} | {nnz:8d} | {len(losses_l):6d}')

# Show sparsity pattern at lambda=0.5
w_sparse, _ = lasso_ista(X_train, y_train, 0.5)
print(f'\nLasso (lambda=0.5) recovered weights:')
print(f'  True nonzero indices: {list(np.where(np.abs(true_w) > 0)[0])}')
print(f'  Lasso nonzero indices: {list(np.where(np.abs(w_sparse) > 1e-6)[0])}')
print(f'  Lasso weights: {np.round(w_sparse[np.abs(w_sparse) > 1e-6], 3)}')

# =====================================================================
# 3. Early stopping as implicit regularization
# =====================================================================
print('\n' + '=' * 65)
print('EARLY STOPPING = IMPLICIT L2 REGULARIZATION')
print('=' * 65)

rng2 = np.random.default_rng(99)
n_pts = 30
x_raw = np.sort(rng2.uniform(-1, 1, n_pts))
y_raw = 0.5 * x_raw ** 2 - 0.3 * x_raw + rng2.normal(scale=0.15, size=n_pts)

deg = 15
X_poly = np.column_stack([x_raw ** d for d in range(deg + 1)])
col_norms = np.linalg.norm(X_poly, axis=0, keepdims=True)
col_norms[col_norms < 1e-12] = 1.0
X_poly = X_poly / col_norms

x_test_raw = np.linspace(-1, 1, 200)
X_test_poly = np.column_stack([x_test_raw ** d for d in range(deg + 1)]) / col_norms
y_test_raw = 0.5 * x_test_raw ** 2 - 0.3 * x_test_raw

lr_gd = 0.01
w_gd = np.zeros(deg + 1)
train_losses, test_losses = [], []
checkpoints = {}

for t in range(5001):
    pred_train = X_poly @ w_gd
    train_loss = mse(y_raw, pred_train)
    test_loss = mse(y_test_raw, X_test_poly @ w_gd)
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    if t in [50, 200, 5000]:
        checkpoints[t] = w_gd.copy()
    grad = (2.0 / n_pts) * X_poly.T @ (pred_train - y_raw)
    w_gd = w_gd - lr_gd * grad

best_t = np.argmin(test_losses)
print(f'Best test MSE at iteration {best_t} (test MSE = {test_losses[best_t]:.5f})')
print(f'Final iteration 5000: train MSE = {train_losses[-1]:.5f}, test MSE = {test_losses[-1]:.5f}')
print(f'Early stopping (t={best_t}) vs full training: implicit regularization reduces '
      f'test MSE by {test_losses[-1] - test_losses[best_t]:.5f}')

# Verify early stopping ~ L2: find Ridge lambda matching early-stopped solution
w_early = checkpoints[200]
best_lam_match, best_diff = 0, np.inf
for lam_try in np.logspace(-4, 2, 500):
    w_ridge_try = ridge_fit(X_poly, y_raw, lam_try)
    diff = np.linalg.norm(w_ridge_try - w_early)
    if diff < best_diff:
        best_diff = diff
        best_lam_match = lam_try
print(f'\nEarly-stopped w (t=200) closest to Ridge w at lambda={best_lam_match:.4f} '
      f'(||w_early - w_ridge||={best_diff:.5f})')
print(f'Theoretical prediction: lambda ~ 1/(lr*T) = 1/({lr_gd}*200) = {1/(lr_gd*200):.2f}')

## 🤖 Why This Matters for AI

Regularization is not an academic afterthought — it is *the* mechanism that makes modern
deep learning work at scale:

- **`weight_decay` in AdamW.** The standard optimizer for training transformers and LLMs
  (GPT, BERT, LLaMA) is AdamW, which applies weight decay *directly to the parameters*
  rather than adding an L2 term to the loss. This distinction matters: for Adam with L2
  loss, the adaptive learning rate scales the regularization differently per parameter;
  AdamW decouples weight decay from the gradient, making it a true L2 penalty in
  parameter space. We demonstrate this difference numerically below.

- **Dropout as approximate L2.** For a single linear layer, dropout with rate $p$ on
  the input is equivalent (in expectation) to L2 regularization with
  $\lambda \propto p/(1-p)$. We verify this equivalence.

- **Early stopping in practice.** Every training run uses a validation set to decide
  when to stop — this *is* implicit regularization, and the choice of patience
  hyperparameter controls the effective complexity.

---

In [ ]:
# =====================================================================
# AI APPLICATION: AdamW vs Adam+L2 -- they are NOT the same
# =====================================================================
import torch
import torch.nn as nn

torch.manual_seed(42)
rng = np.random.default_rng(42)

# Synthetic classification task
n_samples, n_features = 500, 30
X_np = rng.normal(size=(n_samples, n_features)).astype(np.float32)
true_w_np = np.zeros(n_features, dtype=np.float32)
true_w_np[:5] = [2.0, -1.5, 1.0, -0.5, 0.8]
logits_np = X_np @ true_w_np
probs_np = 1 / (1 + np.exp(-logits_np))
y_np = (rng.uniform(size=n_samples) < probs_np).astype(np.float32)

X_t = torch.tensor(X_np)
y_t = torch.tensor(y_np)

def train_model(optimizer_name, weight_decay, lr=0.01, n_steps=300):
    """Train logistic regression; return final weights + loss history."""
    model = nn.Linear(n_features, 1, bias=False)
    nn.init.zeros_(model.weight)
    loss_fn = nn.BCEWithLogitsLoss()

    if optimizer_name == 'AdamW':
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'Adam+L2':
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0)

    losses = []
    for step in range(n_steps):
        logits = model(X_t).squeeze()
        loss = loss_fn(logits, y_t)
        if optimizer_name == 'Adam+L2':
            l2_penalty = weight_decay * sum(p_.pow(2).sum() for p_ in model.parameters())
            loss = loss + l2_penalty
        losses.append(loss.item())
        opt.zero_grad()
        loss.backward()
        opt.step()

    return model.weight.detach().numpy().flatten(), losses

wd = 0.1
w_adamw, loss_adamw = train_model('AdamW', wd)
w_adam_l2, loss_adam_l2 = train_model('Adam+L2', wd)

print('=' * 65)
print('AdamW vs Adam+L2: THEY ARE NOT THE SAME')
print('=' * 65)
print(f'Weight decay / L2 coefficient: {wd}')
print(f'\n{"":>20} {"AdamW":>12} {"Adam+L2":>12}')
print('-' * 48)
print(f'{"||w||_2":>20} {np.linalg.norm(w_adamw):12.4f} {np.linalg.norm(w_adam_l2):12.4f}')
print(f'{"||w||_1":>20} {np.sum(np.abs(w_adamw)):12.4f} {np.sum(np.abs(w_adam_l2)):12.4f}')
print(f'{"max |w_j|":>20} {np.max(np.abs(w_adamw)):12.4f} {np.max(np.abs(w_adam_l2)):12.4f}')
print(f'{"Final loss":>20} {loss_adamw[-1]:12.4f} {loss_adam_l2[-1]:12.4f}')
print(f'{"||w_adamw - w_l2||":>20}   {np.linalg.norm(w_adamw - w_adam_l2):.4f}')
print(f'\nThe two optimizers produce DIFFERENT weight vectors even with the same')
print(f'regularization strength -- AdamW decouples weight decay from the adaptive')
print(f'learning rate, making it a true L2 penalty in parameter space.')

# =====================================================================
# Dropout ~ L2 for a linear model
# =====================================================================
print('\n' + '=' * 65)
print('DROPOUT ~ L2 REGULARIZATION (linear model)')
print('=' * 65)

def train_with_dropout(X, y, p_drop, lr=0.05, n_steps=2000):
    model = nn.Linear(n_features, 1, bias=False)
    nn.init.zeros_(model.weight)
    dropout = nn.Dropout(p=p_drop)
    loss_fn = nn.MSELoss()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(n_steps):
        model.train()
        X_dropped = dropout(X)
        pred = model(X_dropped).squeeze()
        loss = loss_fn(pred, y)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    return model.weight.detach().numpy().flatten()

y_reg = torch.tensor((X_np @ true_w_np).astype(np.float32))
p_drop = 0.3
equiv_lambda = p_drop / (1 - p_drop) * 0.05

torch.manual_seed(0)
w_dropout_runs = [train_with_dropout(X_t, y_reg, p_drop, lr=0.01, n_steps=3000) for _ in range(5)]
w_dropout_avg = np.mean(w_dropout_runs, axis=0)

w_ridge_equiv = ridge_fit(X_np, y_reg.numpy(), equiv_lambda)
w_ols_check = np.linalg.lstsq(X_np, y_reg.numpy(), rcond=None)[0]

print(f'Dropout rate: {p_drop}')
print(f'Theoretical equivalent Ridge lambda proportional to p/(1-p) = {p_drop/(1-p_drop):.3f}')
print(f'\n{"":>20} {"OLS":>12} {"Dropout":>12} {"Ridge":>12}')
print('-' * 60)
print(f'{"||w||_2":>20} {np.linalg.norm(w_ols_check):12.4f} '
      f'{np.linalg.norm(w_dropout_avg):12.4f} {np.linalg.norm(w_ridge_equiv):12.4f}')
print(f'\nDropout shrinks weights toward zero, similar to Ridge (L2) regularization.')
print(f'Both produce smaller ||w||_2 than OLS, confirming the regularization effect.')

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For the Ridge regression problem with $X = \begin{pmatrix} 1 & 0 \\ 0 & 2 \\ 1 & 1 \end{pmatrix}$,
   $y = \begin{pmatrix} 1 \\ 4 \\ 2 \end{pmatrix}$, and $\lambda = 0.5$, compute the Ridge solution
   $\hat{w}_\lambda = (X^\top X + n\lambda I)^{-1}X^\top y$ by hand. Verify with NumPy.

<details>
<summary>💡 Solution</summary>

$X^\top X = \begin{pmatrix} 2 & 1 \\ 1 & 5 \end{pmatrix}$, $X^\top y = \begin{pmatrix} 3 \\ 10 \end{pmatrix}$,
$n\lambda I = 1.5 I = \begin{pmatrix} 1.5 & 0 \\ 0 & 1.5 \end{pmatrix}$.

So $X^\top X + n\lambda I = \begin{pmatrix} 3.5 & 1 \\ 1 & 6.5 \end{pmatrix}$, with inverse
$\frac{1}{3.5 \cdot 6.5 - 1}\begin{pmatrix} 6.5 & -1 \\ -1 & 3.5 \end{pmatrix}
= \frac{1}{21.75}\begin{pmatrix} 6.5 & -1 \\ -1 & 3.5 \end{pmatrix}$.

$\hat{w}_\lambda = \frac{1}{21.75}\begin{pmatrix} 6.5 & -1 \\ -1 & 3.5 \end{pmatrix}
\begin{pmatrix} 3 \\ 10 \end{pmatrix} = \frac{1}{21.75}\begin{pmatrix} 9.5 \\ 32 \end{pmatrix}
\approx \begin{pmatrix} 0.4368 \\ 1.4713 \end{pmatrix}$.

Verification: `X = np.array([[1,0],[0,2],[1,1]]); y = np.array([1,4,2]); lam = 0.5; np.linalg.solve(X.T@X + 3*lam*np.eye(2), X.T@y)` gives `[0.4368, 1.4713]`. ✓

</details>

### 💭 UNDERSTAND
2. Explain geometrically why L1 regularization produces sparse solutions but L2 does not.
   Consider the contour plot of the squared-error loss in 2D weight space, intersected with
   the L1 constraint region ($|w_1|+|w_2| \le t$, a diamond) versus the L2 constraint region
   ($w_1^2 + w_2^2 \le t$, a circle). Why does the diamond's corners make exact zeros likely?

<details>
<summary>💡 Solution</summary>

The loss contours are ellipses centered at the OLS solution $\hat{w}_{\text{OLS}}$.
The regularized solution is where the smallest loss contour touches the constraint region.

For L2, the constraint region is a smooth circle. The tangent point between an ellipse and a
circle will generically lie at an arbitrary angle — not on a coordinate axis — so $w_1$ and
$w_2$ are both nonzero.

For L1, the constraint region is a diamond with corners on the axes (at $(\pm t, 0)$ and
$(0, \pm t)$). Unless the loss contours are aligned in a very specific way, the first
contact point will be at a corner — where one coordinate is exactly zero. In higher
dimensions, the L1 ball has exponentially many corners (one per sign pattern), and each
corner corresponds to a sparse solution. This is why L1 promotes sparsity structurally,
not by accident.

</details>

### ✏️ DERIVE
3. **MAP = Ridge.** Starting from Bayes' theorem with a Gaussian likelihood
   $p(y|X,w) \propto \exp\left(-\frac{1}{2\sigma^2}\|Xw-y\|_2^2\right)$ and Gaussian prior
   $p(w) = \mathcal{N}(0, \tau^2 I)$, show that the MAP estimate
   $\hat{w}_{\text{MAP}} = \arg\max_w [\log p(y|X,w) + \log p(w)]$ is identical to the
   Ridge solution with $\lambda = \sigma^2/(n\tau^2)$.

<details>
<summary>💡 Solution</summary>

The log-posterior (up to constants) is:

$$\log p(w|\mathcal{D}) = -\frac{1}{2\sigma^2}\|Xw-y\|_2^2 - \frac{1}{2\tau^2}\|w\|_2^2 + \text{const}.$$

Maximizing this is equivalent to minimizing:

$$\frac{1}{2\sigma^2}\|Xw-y\|_2^2 + \frac{1}{2\tau^2}\|w\|_2^2.$$

Multiplying through by $2\sigma^2/n$ (which doesn't change the argmin):

$$\frac{1}{n}\|Xw-y\|_2^2 + \frac{\sigma^2}{n\tau^2}\|w\|_2^2.$$

This is exactly the Ridge objective $\frac{1}{n}\|Xw-y\|_2^2 + \lambda\|w\|_2^2$ with
$\lambda = \sigma^2/(n\tau^2)$. ∎

Note: large prior variance $\tau^2$ → small $\lambda$ (weak regularization, diffuse prior);
small $\tau^2$ → large $\lambda$ (strong regularization, tight prior around zero).

</details>

### 🐍 IMPLEMENT
4. Implement FISTA (Fast ISTA) — the accelerated proximal gradient method for Lasso — which adds
   a momentum term to ISTA. Compare convergence speed (objective value vs. iteration) of ISTA
   and FISTA on the same problem. FISTA should achieve $O(1/t^2)$ convergence vs. ISTA's $O(1/t)$.

<details>
<summary>💡 Solution</summary>

FISTA adds Nesterov momentum:
```python
def lasso_fista(X, y, lam, lr=None, max_iter=2000, tol=1e-6):
    n, p = X.shape
    if lr is None:
        L = 2 * np.linalg.norm(X.T @ X / n, ord=2)
        lr = 1.0 / L
    w = np.zeros(p)
    w_prev = w.copy()
    t_k = 1.0
    losses = []
    for k in range(max_iter):
        t_k_next = (1 + np.sqrt(1 + 4 * t_k**2)) / 2
        momentum = (t_k - 1) / t_k_next
        v = w + momentum * (w - w_prev)  # extrapolation step
        grad = (2.0 / n) * X.T @ (X @ v - y)
        z = v - lr * grad
        w_new = np.sign(z) * np.maximum(np.abs(z) - lr * lam, 0)
        obj = (1/n) * np.sum((X @ w_new - y)**2) + lam * np.sum(np.abs(w_new))
        losses.append(obj)
        w_prev = w
        w = w_new
        t_k = t_k_next
    return w, losses
```
FISTA reaches a given tolerance in roughly $\sqrt{\text{ISTA iterations}}$ steps.

</details>

### 🤖 APPLY
5. Train a small neural network (e.g., a 3-layer MLP) on a synthetic 2D classification
   problem with and without L2 regularization (use `weight_decay` in the optimizer). Plot the
   decision boundary for both models, and compute the train/test accuracy gap. The regularized
   model should show a smoother decision boundary and a smaller generalization gap.

<details>
<summary>💡 Solution</summary>

Generate 2D data with `sklearn.datasets.make_moons` or a custom spiral. Train a
`nn.Sequential(nn.Linear(2,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,1))`
with `torch.optim.AdamW(params, lr=1e-3, weight_decay=0)` (no reg) and
`weight_decay=0.1` (with reg). Plot decision boundaries on a meshgrid. The unregularized
model will show jagged, overfitting boundaries especially in regions with sparse data;
the regularized model will be smoother. Typical accuracy gaps: unregularized might show
train=99%, test=88%; regularized: train=95%, test=92%.

</details>

### 🚀 CHALLENGE
6. **AdamW ≠ Adam + L2 loss.** Derive mathematically why `torch.optim.AdamW` with
   `weight_decay=λ` produces a *different* update rule from `torch.optim.Adam` with an
   L2 penalty $\lambda\|w\|_2^2$ added to the loss. Write out both update rules step by
   step, identifying where the adaptive second-moment estimate $v_t$ interacts differently
   with the weight decay term. Then verify empirically that the two produce different
   final weights on a nontrivial problem.

<details>
<summary>💡 Solution</summary>

**Adam + L2 loss:** The gradient of $\mathcal{L} + \lambda\|w\|^2$ is
$g_t = \nabla\mathcal{L} + 2\lambda w_t$. This modified gradient is fed into Adam's
moment estimates:
$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t$,
$v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$.
The update is $w_{t+1} = w_t - \eta \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$.

The $2\lambda w_t$ term is *inside* the adaptive scaling — parameters with large
accumulated gradients have their weight decay effectively reduced.

**AdamW:** Computes moments from the *unregularized* gradient $g_t = \nabla\mathcal{L}$
only, then applies weight decay *after* the adaptive step:
$w_{t+1} = (1 - \eta\lambda)w_t - \eta \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$.

The weight decay is *decoupled* from the adaptive learning rate — every parameter
is decayed by the same factor $(1-\eta\lambda)$ regardless of its gradient history.
This makes AdamW a true L2 regularizer in parameter space, while Adam+L2 applies
an adaptively scaled (and therefore inconsistent) penalty. See Loshchilov & Hutter
(2019) for the original analysis.

</details>

---

## 📚 Further Reading
- Vapnik, *Statistical Learning Theory* (1998), Ch. 4 (Structural Risk Minimization)
- Hastie, Tibshirani & Friedman, *Elements of Statistical Learning*, Ch. 3 (Linear Methods) and Ch. 7 (Model Selection)
- Loshchilov & Hutter, ["Decoupled Weight Decay Regularization"](https://arxiv.org/abs/1711.05101) (2019) — the AdamW paper
- Shalev-Shwartz & Ben-David, *Understanding Machine Learning*, Ch. 7 (The Bias-Complexity Tradeoff) and Ch. 13 (Regularization)
- Tibshirani, ["Regression Shrinkage and Selection via the Lasso"](https://doi.org/10.1111/j.2517-6161.1996.tb02080.x) (1996)

---

*Next notebook: `13.04` PAC Learning and Sample Complexity*